# HomeGuardian Mi A3 - native Linux remote build environment probe

Phase 3 of the owner directive: establish what this remote runner can actually do before
moving kernel/ARM64 work off the workstation. Read-only probe; no device, no private media,
no credentials.


In [ ]:
import os, platform, shutil, subprocess, json, sys, textwrap

def sh(cmd, timeout=120):
    try:
        p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, timeout=timeout)
        return p.returncode, (p.stdout or "").strip(), (p.stderr or "").strip()
    except Exception as e:
        return 127, "", str(e)

def first_line(s):
    return s.splitlines()[0] if s else ""

def meminfo():
    out = {}
    try:
        for line in open("/proc/meminfo"):
            k = line.split(":")[0]
            if k in ("MemTotal", "MemAvailable", "SwapTotal"):
                out[k + "_MB"] = int(line.split()[1]) // 1024
    except Exception:
        pass
    return out

env = {}
env["architecture"] = platform.machine()
env["platform"] = platform.platform()
env["cpus"] = os.cpu_count()
env.update(meminfo())
env["disk"] = first_line(sh("df -h /kaggle/working")[1])
env["disk_all"] = sh("df -h")[1]
env["toolchain"] = {
    "gcc": first_line(sh("gcc --version")[1]),
    "g++": first_line(sh("g++ --version")[1]),
    "cmake": first_line(sh("cmake --version")[1]),
    "make": first_line(sh("make --version")[1]),
    "git": first_line(sh("git --version")[1]),
    "python": sys.version.split()[0],
}
env["internet"] = sh("getent hosts github.com")[0] == 0
env["apt_available"] = shutil.which("apt-get") is not None
env["sudo"] = sh("sudo -n true")[0] == 0

# --- the question that decides the whole ARM64 strategy on an x86_64 runner ---
env["qemu"] = {b: shutil.which(b) for b in
               ["qemu-aarch64-static", "qemu-aarch64", "qemu-img"]}
env["cross_compilers"] = {b: shutil.which(b) for b in
    ["aarch64-linux-gnu-gcc", "aarch64-linux-gnu-g++", "arm-linux-gnueabihf-gcc"]}
env["docker"] = shutil.which("docker")
env["kvm"] = os.path.exists("/dev/kvm")
env["binfmt"] = os.path.exists("/proc/sys/fs/binfmt_misc")

# --- GPU truth: metadata is a request, not a fact ---
rc, out, _ = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
env["gpu"] = {"exit_code": rc, "devices": out.splitlines()} if rc == 0 else {"exit_code": rc, "devices": []}
env["cuda_visible_devices"] = os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>")

# --- does the filesystem permit a build (exec bit on a real temp dir)? ---
probe = "/kaggle/working/.exec_probe.sh"
open(probe, "w").write("#!/bin/sh\necho exec_ok\n")
os.chmod(probe, 0o755)
env["exec_permitted"] = sh(probe)[1] == "exec_ok"
os.remove(probe)
env["free_space_working_MB"] = int(sh("df -Pm /kaggle/working")[1].splitlines()[-1].split()[3]) if sh("df -Pm /kaggle/working")[0] == 0 else None

print(json.dumps(env, indent=2))
open("/kaggle/working/mi_a3_env_probe.json", "w").write(json.dumps(env, indent=2))
print("PROBE OK")


In [ ]:
# --- explicit feasibility verdicts, so a reader does not have to infer them ---
from textwrap import dedent
arch = env["architecture"]
have_cross = any(env["cross_compilers"].values())
have_qemu = env["qemu"].get("qemu-aarch64-static") or env["qemu"].get("qemu-aarch64")
print(dedent(f"""
RUNNER               : {arch} / {env['cpus']} cpus / {env.get('MemAvailable_MB')} MB available
ARM64 STRATEGY       : {'cross-compile available now' if have_cross else ('install aarch64 cross toolchain via apt (internet=' + str(env['internet']) + ')')}
QEMU USER-MODE       : {'present at ' + str(have_qemu) if have_qemu else 'ABSENT - arm64 binaries cannot execute here yet'}
NESTED VIRT / KVM    : {'/dev/kvm present' if env['kvm'] else 'no /dev/kvm - no VM-based testing'}
DOCKER               : {env['docker'] or 'absent - cannot reuse the container build method here'}
GPU (actual device)  : {env['gpu']['devices'] or 'none allocated'}
BUILD FS PERMITS EXEC: {env['exec_permitted']}
WORKING FREE SPACE   : {env.get('free_space_working_MB')} MB
""").strip())
